<span style="font-family: 'Courier New', monospace;">

*AI-generated draft (Claude, Anthropic) — for review. All parameters derive from version-controlled scripts and data.*

# 40 — Wednesday validation sort (BLIND) for the Scene-1 classifier

Sort the **96-recording stratified Wednesday sample** (`scene_sorting/wednesday_validation/`, built by `build_wednesday_validation.py`, 3 per year×slot). This is an **independent check** that the trained Scene-1 classifier — and thus the M-F harvest's recall, which was only validated against manual *Monday* — actually transfers to Wednesdays.

**⚠️ BLIND — do not peek.** The classifier's predictions are already computed and locked in at `validation/scene_classifier_gate/wednesday_validation/predictions.csv` (committed before this sort). **Do NOT open that file.** Sort each sheet by your own rubric only; if you see the model's guess first it anchors your calls and the validation is worthless.

**Rubric (`scene_sorting/scene1_rubric.md`):** Scene 1 = the Mushroom sulfide **TOWER, face-on, zoomed in, centered/upright, vent orifice visible**. NOT wide/survey, edge-on/behind, or generic vent-community texture.

**How to use** (same widget-free engine as nb25 — no ipywidgets, RTC-safe):
1. Run the config cell + the engine cell once (top to bottom). The first sheet appears.
2. In the **driver cell**, call one of these then press **Ctrl+Enter** (NOT Shift+Enter):
   - `s1(t)` — Scene 1, front-on view appears ~**t seconds** (read the `t=...s` tile label)
   - `no()` — Not Scene 1   ·   `sk()` — skip   ·   `undo()` — fix a misclick   ·   `show()` — redraw
3. Reuse the one driver cell for every sheet. Resumable — filed sheets leave `contact_sheets/`.

**When all 96 are sorted, tell Claude.** Claude re-runs `gate_scene_classifier.py`, which compares your blind calls to the locked predictions and reports **Cohen's kappa, raw agreement (Wilson 95% CI), the disagreement list, and Scene-1 timing agreement** → `gate_results.md`.

</span>

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "scripts"))

import matplotlib.pyplot as plt
import scene_sorter as sorter

# --- CONFIG: the blind Wednesday validation sample (96 recordings) ----------
# Sort WITHOUT looking at the classifier's predictions.csv in this folder.
# When done, Claude runs scripts/gate_scene_classifier.py --session <this dir>
# to score your blind calls vs the locked predictions (kappa / agreement / timing).
SESSION_DIR = Path.cwd().parent / "scene_sorting" / "wednesday_validation"

print("Sorting:", SESSION_DIR)
print("Status :", sorter.counts(SESSION_DIR))


In [ ]:
# --- Widget-free sorting engine --------------------------------------------
import csv

_skipped: set[str] = set()  # sheet filenames skipped this session


def _pending():
    return [p for p in sorter.pending_sheets(SESSION_DIR) if p.name not in _skipped]


def show():
    """Redraw the current (next-to-sort) contact sheet, full size."""
    plt.close("all")
    rem = _pending()
    c = sorter.counts(SESSION_DIR)
    print(
        f"Scene 1: {c['scene1']}   |   Not Scene 1: {c['not_scene1']}   |   "
        f"remaining: {len(rem)}   (skipped this session: {len(_skipped)})"
    )
    if not rem:
        print("\u2705 Nothing left to sort in this session.")
        return
    sheet = rem[0]
    _fig, ax = plt.subplots(figsize=(16, 9))
    ax.imshow(plt.imread(sheet))
    ax.axis("off")
    ax.set_title(sheet.stem, fontsize=11)
    plt.show()


def _decide(decision, scene1_time_s=None):
    rem = _pending()
    if not rem:
        show()
        return
    sheet = rem[0]
    if decision == "skip":
        _skipped.add(sheet.name)
    sorter.apply_decision(SESSION_DIR, sheet, decision, scene1_time_s=scene1_time_s)
    if decision == "scene1" and scene1_time_s is None:
        print(
            "\u26a0 Scene 1 filed with NO time \u2014 this recording can't be counted "
            "until a time is recorded. Run undo() then s1(<t>) to add it."
        )
    show()


def s1(t=None):
    """File the current sheet as Scene 1 at clip-time ``t`` seconds, then show the next.

    Read ``t`` from the ``t=...s`` label under the tile showing the front-on view,
    e.g. ``s1(270)``. Omit only if you can't locate it (logged blank).
    """
    _decide("scene1", scene1_time_s=t)


def no():
    """File the current sheet as Not Scene 1, then show the next."""
    _decide("not_scene1")


def sk():
    """Skip the current sheet (leave it in place), then show the next."""
    _decide("skip")


def undo():
    """Move the most recently filed sheet back to contact_sheets/ for re-review."""
    paths = sorter.session_paths(SESSION_DIR)
    if not paths.log_csv.exists():
        print("No decisions logged yet \u2014 nothing to undo.")
        return
    rows = list(csv.DictReader(paths.log_csv.open()))
    filed = [r for r in rows if r["decision"] in sorter.FILED_DECISIONS]
    if not filed:
        print("Nothing filed yet \u2014 nothing to undo.")
        return
    last = filed[-1]
    name = last["stem"] + ".png"
    src = paths.target_for(last["decision"]) / name
    if not src.exists():
        print(f"Cannot undo \u2014 {name} is not in {last['decision']}/ (already undone?).")
        return
    src.rename(paths.contact_sheets / name)
    _skipped.discard(name)
    print(f"Moved back: {name}  (was filed as {last['decision']})")
    show()


print(
    "Engine ready.  s1(t) = Scene 1 at t seconds   no() = Not Scene 1   sk() = Skip\n"
    "Edit the call in the driver cell below and press Ctrl+Enter for each sheet."
)
show()

In [ ]:
# DRIVER CELL — look at the sheet above, set your call, press Ctrl+Enter.
#   s1(t) = Scene 1 at t seconds (read the t=...s label)  |  no() = Not Scene 1
#   sk()  = Skip   |   undo() = fix last misclick   |   show() = redraw current
#
# Reuse THIS ONE cell for every decision — edit the call and Ctrl+Enter (run in place).
# Use Ctrl+Enter, NOT Shift+Enter: on this last cell Shift+Enter inserts a NEW cell each
# time, re-embedding the ~3 MB sheet per sort until the notebook is too big to open.

no()

<span style="font-family: 'Courier New', monospace;">

*AI-generated note (Claude, Anthropic).*

## ⬇️ STOP — do not sort below this line

This cell exists on purpose. Pressing **Shift+Enter** on the driver cell above now
lands you **here** instead of creating a new cell, so the notebook can't bloat.
**Scroll back up to the driver cell** and reuse it for the next decision
(edit the call, **Ctrl+Enter**). Never sort in a new cell below — that is what made
the notebook grow too large to reopen.
</span>